In [ ]:
from dolfinx import log, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
import pyvista
import numpy as np
import ufl

from mpi4py import MPI
from dolfinx import fem, mesh, plot

from pathlib import Path
from dolfinx import io

import pickle


log.set_log_level(log.LogLevel.ERROR)
# import uniaxialGeometry
# domain, facet_tags, cell_tags = uniaxialGeometry.create_uniaxial_geometry(h=3)
import uniaxialGeometryHex
domain, facet_tags, cell_tags = uniaxialGeometryHex.create_uniaxial_geometry(h=3)
L0 = 50

In [ ]:
# import pyvista

# cells, types, x = plot.vtk_mesh(domain)
# grid = pyvista.UnstructuredGrid(cells, types, x)
# plotter = pyvista.Plotter()
# plotter.add_mesh(grid, show_edges=True)
# plotter.show()

# Weak form

In [ ]:
import basix.ufl
el_u = basix.ufl.element("Lagrange", domain.basix_cell(), 2, shape=(domain.geometry.dim,))
el_p = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
el_mixed = basix.ufl.mixed_element([el_u, el_p])
W = fem.functionspace(domain, el_mixed)

w = fem.Function(W)

In [ ]:

V_u = W.sub(0) # displacement subspace 
V_uCollapsed, V_uCollapsed_to_Vu = V_u.collapse()

u_D_left = fem.Function(V_uCollapsed)
left_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tags.find(10))
# left_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim, entities=cell_tags.find(1))

u_D_right = fem.Function(V_uCollapsed)
right_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tags.find(11))
# right_dofs = fem.locate_dofs_topological(V=(V_u, V_uCollapsed), entity_dim=domain.topology.dim, entities=cell_tags.find(5))

bcs = [fem.dirichletbc(u_D_left, left_dofs, V_u), fem.dirichletbc(u_D_right, right_dofs, V_u)]

In [ ]:
B = fem.Constant(domain, default_scalar_type((0, 0, 0)))
T = fem.Constant(domain, default_scalar_type((0, 0, 0)))

In [ ]:
(u, p) = ufl.split(w)
v_u, v_p = ufl.TestFunctions(W)


In [ ]:
d = len(u) # Spatial dimension
I = ufl.variable(ufl.Identity(d)) # Identity tensor
F = ufl.variable(I + ufl.grad(u)) # Deformation gradient
C = ufl.variable(F.T * F) # Right Cauchy-Green tensor

# Invariants of deformation tensors, see https://en.wikipedia.org/wiki/Invariants_of_tensors
I_1 = ufl.variable(ufl.tr(C))
J = ufl.variable(ufl.det(F))
I_1_bar = ufl.variable(J**(-2/3) * I_1) 
I_2 = 0.5 * (ufl.tr(C) ** 2 - ufl.tr(C * C))

In [ ]:
# Neo Hook Incompressible
# E = default_scalar_type(1.0e4)
# nu = default_scalar_type(0.3)
# mu = fem.Constant(domain, E / (2 * (1 + nu)))
# lmbda = fem.Constant(domain, E * nu / ((1 + nu) * (1 - 2 * nu)))
# psi = (mu / 2) * (I_1_bar - 3) + p*(J-1) 

# Simplified Mooney Rivlin TP1
c_10 = -11.11
c_01 = 17.4
c_02 = 3.134

# Simplified Mooney Rivlin TP3
c_10 = -16.5
c_01 = 26.36
c_02 = 4.524

C_10 = fem.Constant(domain, default_scalar_type(c_10))
C_01 = fem.Constant(domain, default_scalar_type(c_01))
C_02 = fem.Constant(domain, default_scalar_type(c_02))

psi = C_10 * (I_1 - 3) + C_01 * (I_2 - 3) + C_02 * (I_2 - 3) ** 2 + p*(J-1)

P = ufl.diff(psi, F)

sigma = P * 1/J * F.T

In [ ]:
# Define the variational form with traction integral over all facets with value 2. We set the quadrature degree for the integrals to 4.
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags, metadata={"quadrature_degree": 4})
dx = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": 4})

In [ ]:
residual = (
    ufl.inner(ufl.grad(v_u), P) * dx - ufl.inner(v_u, B) * dx - ufl.inner(v_u, T) * ds(2) + ufl.inner((J-1), v_p)*dx
)

# Solving

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_monitor": None,
    "snes_atol": 1e-8,
    "snes_rtol": 1e-8,
    "snes_stol": 1e-8,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
problem = NonlinearProblem(
    residual,
    w,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="hyperelasticity",
)

In [ ]:
log.set_log_level(log.LogLevel.ERROR)


currentDisplacement = 0
finalDisplacement = 100 
maxDisplacementStep = 5 
minDisplacementStep = 0.05
t = 0 
lastLoopSucceeded = True 
displacementStep = maxDisplacementStep

folder = Path("results")
folder.mkdir(exist_ok=True, parents=True)
xdmf = io.XDMFFile(MPI.COMM_WORLD, folder/"NiklasTesting2.xdmf", "w")
xdmf.write_mesh(domain)
xdmf.write_meshtags(facet_tags, domain.geometry)
xdmf.write_meshtags(cell_tags, domain.geometry)
import pandas as pd 
stressStrainCurve = []


In [ ]:
def attemptStep(t, displacement): 
    # backup current state 
    wSave = fem.Function(W)
    wSave.x.array[:]=w.x.array[:]

    # attempt solving 
    print(f"Attempting time step {t}, Displacement {displacement}")
    u_D_right.x.array[0::3] = displacement
    problem.solve()
    converged = problem.solver.getConvergedReason()
    num_its = problem.solver.getIterationNumber()
    print(f"Solver convergence: {converged}. Number of iterations {num_its}")
    if converged < 0: 
        print(f"Solver did not converge on time step {t}, Displacement {displacement}")
        # reset to backup 
        w.x.array[:] = wSave.x.array[:]
        return False 
    else: 
        return True 

In [ ]:
def writeResults():
    # write results and all that 
    # u 
    V_u_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,)))
    u_out = fem.Function(V_u_out)
    u_out.name = "u"
    u_out.interpolate(w.sub(0).collapse())
    xdmf.write_function(u_out, t)

    # p
    V_p_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1))
    p_out = fem.Function(V_p_out)
    p_out.name = "p"
    p_out.interpolate(w.sub(1).collapse())
    xdmf.write_function(p_out, t)

    # J
    V_J_post = fem.functionspace(domain, ("Lagrange", 1))
    J_post = fem.Expression(J, V_J_post.element.interpolation_points)
    J_out = fem.Function(V_J_post)
    J_out.name = "J"
    J_out.interpolate(J_post)
    xdmf.write_function(J_out, t)

    # P 
    V_P_post = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    P_post = fem.Expression(P, V_P_post.element.interpolation_points)
    P_out = fem.Function(V_P_post)
    P_out.name = "P"
    P_out.interpolate(P_post)
    xdmf.write_function(P_out, t)

    # sigma 
    V_sigma_post = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    sigma_post = fem.Expression(sigma, V_sigma_post.element.interpolation_points)
    sigma_out = fem.Function(V_sigma_post)
    sigma_out.name = "sigma"
    sigma_out.interpolate(sigma_post)
    xdmf.write_function(sigma_out, t)

    # get nominal strain
    L_leftEdgeDofs = fem.locate_dofs_topological(V=V_u_out, entity_dim=domain.topology.dim-1, entities=facet_tags.find(20))
    L_leftBorder_X = np.mean(u_out.x.array[0::3][L_leftEdgeDofs])
    L_rightEdgeDofs = fem.locate_dofs_topological(V=V_u_out, entity_dim=domain.topology.dim-1, entities=facet_tags.find(21))
    L_rightBorder_X = np.mean(u_out.x.array[0::3][L_rightEdgeDofs])
    L = L0 + L_rightBorder_X - L_leftBorder_X # since u only contains displacement not position we need to add L0 to get current L
    nominalStrain = (L-L0)/L0 

    # get nominal stress in MPa (max? avg?)
    centerDofs = fem.locate_dofs_topological(V=V_P_post, entity_dim=domain.topology.dim, entities=cell_tags.find(3))
    nominalStress = np.mean(P_out.x.array[0::9][centerDofs])

    # into dataframe
    stressStrainCurve.append([nominalStrain, nominalStress, currentDisplacement])

In [ ]:
# init with displacement 0 
attemptStep(0, 0)
writeResults()
t +=1 

# for t, displacement in enumerate(displacements):
while currentDisplacement < finalDisplacement: 

    if lastLoopSucceeded: 
        displacementStep = min(displacementStep*2, maxDisplacementStep)
    else: 
        displacementStep = displacementStep / 2 

    if displacementStep < minDisplacementStep: 
        print(f"Aborted at {currentDisplacement} mm displacement because stepsize go too small.")
        break 

    success = attemptStep(t, currentDisplacement + displacementStep)

    if success: 
        lastLoopSucceeded = True 
        currentDisplacement = currentDisplacement + displacementStep
        t += 1 
        writeResults()

    else: 
        lastLoopSucceeded = False 




In [ ]:

stressStrainCurveDf = pd.DataFrame(columns=["nominalStrain", "nominalStress", "displacement"], data=stressStrainCurve)

with open(f"{folder}/niklasTesting2StressStrain.pickle", 'wb') as f:
    # Pickle the 'data' dictionary using the highest protocol available.
    pickle.dump(stressStrainCurveDf, f, pickle.HIGHEST_PROTOCOL)

xdmf.close()

In [ ]:
stressStrainCurveDf